In [1]:
import bw2data, bw2io, bw2calc
from bw_timex import TimexLCA
from bw_temporalis import TemporalDistribution, easy_timedelta_distribution
import numpy as np
from datetime import datetime
import os
import re
import pandas as pd
import numpy as np
import pickle

In [2]:
import sys
sys.path.append('../utils/') 
from elec_builder import *

In [3]:
# activate the bw project
bw2data.projects.set_current("ei311")

In [4]:
bw2data.databases

Databases dictionary with 17 object(s):
	ecoinvent-3.11-biosphere
	ecoinvent-3.11-cutoff
	ei_cutoff_3.11_image_SSP1-VLLO_2030 2025-11-24
	ei_cutoff_3.11_image_SSP1-VLLO_2040 2025-11-22
	ei_cutoff_3.11_image_SSP1-VLLO_2050 2025-11-22
	ei_cutoff_3.11_image_SSP2-M_2030 2025-11-22
	ei_cutoff_3.11_image_SSP2-M_2040 2025-11-22
	ei_cutoff_3.11_image_SSP2-M_2050 2025-11-22
	ei_cutoff_3.11_image_SSP5-H_2030 2025-11-22
	ei_cutoff_3.11_image_SSP5-H_2040 2025-11-22
	ei_cutoff_3.11_image_SSP5-H_2050 2025-11-22
	ei_cutoff_3.11_remind_SSP2-NDC_2030 2025-11-21
	ei_cutoff_3.11_remind_SSP2-NDC_2040 2025-11-21
	ei_cutoff_3.11_remind_SSP2-NDC_2050 2025-11-21
	elec_PV_foreground
	elec_hydro_reservoir_foreground
	elec_wind_foreground

### 1. build up foreground (static)
#### for CN reservoir, using RoW as proxy
#### for US reservoir, using CA-QC as proxy, it's already replaced with Annie's CH4 / CO2 values in AB

##### build only ONE ACT in each database 

#### still one final database to check if BW2_timex differs when using ONE-ACT-only Foreground vs. many-ACT Foreground

In [6]:
hydro_all = ["CA-QC", 'RoW']

xx = build_dynamic_electricity_all(
    locations = hydro_all, 
    pathways = [ "SSP1-VLLO" , "SSP2-M", "SSP5-H" ],
    years = [2030, 2040, 2050], 
    elec_act = 'electricity production, hydro, reservoir, non-alpine region',  
    ref_name = 'market for electricity, hydro, high voltage',
    fg_db_name="elec_hydro_reservoir_foreground",
    flush_fg_db = True
)

Flushing existing foreground DB: 'elec_hydro_reservoir_foreground'
Created fresh foreground DB: 'elec_hydro_reservoir_foreground'


In [7]:
hydro_db = bw2data.Database("elec_hydro_reservoir_foreground")
list(hydro_db)

['market for electricity, hydro, high voltage, RoW, SSP2-M, 2040' (kWh, RoW, None),
 'market for electricity, hydro, high voltage, RoW, SSP5-H, 2030' (kWh, RoW, None),
 'market for electricity, hydro, high voltage, RoW, SSP1-VLLO, 2040' (kWh, RoW, None),
 'market for electricity, hydro, high voltage, CA-QC, SSP2-M, 2050' (kWh, CA-QC, None),
 'market for electricity, hydro, high voltage, RoW, SSP2-M, 2050' (kWh, RoW, None),
 'market for electricity, hydro, high voltage, CA-QC, SSP2-M, 2030' (kWh, CA-QC, None),
 'market for electricity, hydro, high voltage, CA-QC, SSP5-H, 2030' (kWh, CA-QC, None),
 'market for electricity, hydro, high voltage, CA-QC, SSP2-M, 2040' (kWh, CA-QC, None),
 'market for electricity, hydro, high voltage, RoW, SSP1-VLLO, 2030' (kWh, RoW, None),
 'market for electricity, hydro, high voltage, CA-QC, SSP1-VLLO, 2040' (kWh, CA-QC, None),
 'market for electricity, hydro, high voltage, CA-QC, SSP5-H, 2050' (kWh, CA-QC, None),
 'market for electricity, hydro, high volta

In [8]:
len(list(hydro_db))

18

In [9]:
rows = []   # collect results for all acts
act_list = list(hydro_db)   
for act in act_list: 
    print(act)

    name = act.get("name")
    name_parts = [p.strip() for p in name.split(",")]
    # run static LCI + premise_GWP vs. pGWP100 first: 
    pgwp_fixedco2 = find_dpGWP100_method(name_parts[-2], int(name_parts[-1]), method_suffix = "pGWP100 - fixed-AGWPCO2")
    pgwp_dpco2 = find_dpGWP100_method(name_parts[-2], int(name_parts[-1]), method_suffix = "pGWP100 - dp-AGWPCO2")
    gwp =  ('ecoinvent-3.11', 'IPCC 2021 (incl. biogenic CO2)', 'climate change: total (incl. biogenic CO2, incl. SLCFs)', 'global warming potential (GWP100)')   

    lca_gwp = bw2calc.lca.LCA({act: 1}, method=gwp)
    lca_gwp.lci(); lca_gwp.lcia()
    score_gwp = float(lca_gwp.score)

    lca_fixed = bw2calc.lca.LCA({act: 1}, method=pgwp_fixedco2)
    lca_fixed.lci(); lca_fixed.lcia()
    score_fixedco2 = float(lca_fixed.score)

    lca_dp = bw2calc.lca.LCA({act: 1}, method=pgwp_dpco2)
    lca_dp.lci(); lca_dp.lcia()
    score_dpco2 = float(lca_dp.score)

    print(score_gwp, score_fixedco2, score_dpco2) 

    # ---- store results for this activity ----
    rows.append({
        "Activity": name,                      # index value later
        "gwp100": score_gwp,
        "pGWP100_fixedCO2": score_fixedco2,
        "pGWP100_dpCO2": score_dpco2,
    })


'market for electricity, hydro, high voltage, CA-QC, SSP2-M, 2030' (kWh, CA-QC, None)
0.027237480987037516 0.03158437928431752 0.02711969689287509
'market for electricity, hydro, high voltage, RoW, SSP5-H, 2030' (kWh, RoW, None)
0.05076620089003004 0.06826572518730606 0.04941494612487993
'market for electricity, hydro, high voltage, CA-QC, SSP5-H, 2040' (kWh, CA-QC, None)
0.015677933911314414 0.020513518225509387 0.015532795585280867
'market for electricity, hydro, high voltage, CA-QC, SSP1-VLLO, 2050' (kWh, CA-QC, None)
0.025149297413734607 0.025654805949944928 0.026053074236294882
'market for electricity, hydro, high voltage, RoW, SSP1-VLLO, 2050' (kWh, RoW, None)
0.047436732219881 0.047269481689540434 0.04806139092343953
'market for electricity, hydro, high voltage, RoW, SSP2-M, 2030' (kWh, RoW, None)
0.050624132429712226 0.0588433577155799 0.05050087185795245
'market for electricity, hydro, high voltage, CA-QC, SSP1-VLLO, 2040' (kWh, CA-QC, None)
0.015027617823721764 0.014985699463

In [10]:
df_scores = pd.DataFrame(rows)
df_scores = df_scores.set_index("Activity")

df2 = df_scores.sort_values(by=['Activity'])
df2

,gwp100,pGWP100_fixedCO2,pGWP100_dpCO2
Activity,,,
"market for electricity, hydro, high voltage, CA-QC, SSP1-VLLO, 2030",0.027212,0.028281,0.027695
"market for electricity, hydro, high voltage, CA-QC, SSP1-VLLO, 2040",0.015028,0.014986,0.015086
"market for electricity, hydro, high voltage, CA-QC, SSP1-VLLO, 2050",0.025149,0.025655,0.026053
"market for electricity, hydro, high voltage, CA-QC, SSP2-M, 2030",0.027237,0.031584,0.027120
"market for electricity, hydro, high voltage, CA-QC, SSP2-M, 2040",0.015562,0.017235,0.015566
"market for electricity, hydro, high voltage, CA-QC, SSP2-M, 2050",0.026952,0.028855,0.027409
"market for electricity, hydro, high voltage, CA-QC, SSP5-H, 2030",0.027293,0.035153,0.025494
"market for electricity, hydro, high voltage, CA-QC, SSP5-H, 2040",0.015678,0.020514,0.015533
"market for electricity, hydro, high voltage, CA-QC, SSP5-H, 2050",0.027074,0.032042,0.025919


In [11]:
df2.index = df2.index.str.replace("RoW", "CN", regex=False)
df2.index = df2.index.str.replace("CA-QC", "US", regex=False)
df2

,gwp100,pGWP100_fixedCO2,pGWP100_dpCO2
Activity,,,
"market for electricity, hydro, high voltage, US, SSP1-VLLO, 2030",0.027212,0.028281,0.027695
"market for electricity, hydro, high voltage, US, SSP1-VLLO, 2040",0.015028,0.014986,0.015086
"market for electricity, hydro, high voltage, US, SSP1-VLLO, 2050",0.025149,0.025655,0.026053
"market for electricity, hydro, high voltage, US, SSP2-M, 2030",0.027237,0.031584,0.027120
"market for electricity, hydro, high voltage, US, SSP2-M, 2040",0.015562,0.017235,0.015566
"market for electricity, hydro, high voltage, US, SSP2-M, 2050",0.026952,0.028855,0.027409
"market for electricity, hydro, high voltage, US, SSP5-H, 2030",0.027293,0.035153,0.025494
"market for electricity, hydro, high voltage, US, SSP5-H, 2040",0.015678,0.020514,0.015533
"market for electricity, hydro, high voltage, US, SSP5-H, 2050",0.027074,0.032042,0.025919


In [12]:
df2.to_excel("dp-LCI_output/staticLCI_(p)GWP100/hydro_reservoir_CN-asRoW_US-asQC_staticLCI_threeGWP100.xlsx")

### 2. building dpLCI

In [13]:
#hydro_db = bw2data.Database("elec_hydro_reservoir_foreground")
len(list(hydro_db))

18

In [14]:
hydro_db_203050 = [
    act for act in hydro_db 
    if "2050" in str(act.get('name', '')) or  "2030" in str(act.get('name', ''))
]
len(list(hydro_db_203050))

12

In [15]:
database_dates = {
    'ei_cutoff_3.11_image_SSP1-VLLO_2030 2025-11-24': datetime.strptime("2030", "%Y"),
    'ei_cutoff_3.11_image_SSP1-VLLO_2040 2025-11-22': datetime.strptime("2040", "%Y"),
    'ei_cutoff_3.11_image_SSP1-VLLO_2050 2025-11-22': datetime.strptime("2050", "%Y"),

    'ei_cutoff_3.11_image_SSP2-M_2030 2025-11-22': datetime.strptime("2030", "%Y"),
    'ei_cutoff_3.11_image_SSP2-M_2040 2025-11-22': datetime.strptime("2040", "%Y"),
    'ei_cutoff_3.11_image_SSP2-M_2050 2025-11-22': datetime.strptime("2050", "%Y"),

    'ei_cutoff_3.11_image_SSP5-H_2030 2025-11-22': datetime.strptime("2030", "%Y"),
    'ei_cutoff_3.11_image_SSP5-H_2040 2025-11-22': datetime.strptime("2040", "%Y"),
    'ei_cutoff_3.11_image_SSP5-H_2050 2025-11-22': datetime.strptime("2050", "%Y"),
    
    "elec_hydro_reservoir_foreground": "dynamic", # flag databases that should be temporally distributed with "dynamic"
}

In [16]:
find_dpGWP100_method('SSP1-VLLO', 2030, method_suffix = "pGWP100 - dp-AGWPCO2")

('Climate Change prospective GWP100',
 'SSP119',
 'MY2030',
 'pGWP100 - dp-AGWPCO2')

In [17]:
dp_results = {}

for act in list(hydro_db_203050): 
    print(act)
 
    #### dynamic LCI:: 
    assign_td_from_foreground_db( 
        select_act = act,
        elec_td_year=10,
        resolution="Y",
        kind="uniform",
        fg_db_name="elec_hydro_reservoir_foreground",
        verbose = True
     )

    tlca = run_dp_timex_lca(foreground_act = act ,   
                    pathway = None,
                    year = None,
                    method = None,
                    database_dates = database_dates, #None not working, has to incl. all 9 background DB ... 
                    temporal_grouping="year", 
                    method_prefix = "Climate Change prospective GWP100",
                    method_suffix = "pGWP100 - fixed-AGWPCO2", 
                    fg_db_name = 'elec_hydro_reservoir_foreground'
     )

    tlca.lci()
    tlca.dynamic_inventory.shape

    # dyn-foreground LCI + static background LCI + pGWP100
    lca_0 = tlca.base_score 
    print(lca_0)
    # dpLCI (dyn-foreground LCI + dyn background LCI) +  pGWP100
    tlca.static_lcia()
    lca_1 = tlca.static_score
    print(lca_1)

    act_name = act.get("name")
    dp_results[act_name] = {
        "dyn FG LCI static BG p-LCI, pGWP100-fixedCO2": float(lca_0),   # static BG LCI + dyn FG LCI
        "full dp-LCI, pGWP100-fixedCO2": float(lca_1),      # dyn BG LCI + dyn FG LCI
    }

    #### now save all dyLCI flows to pandas 
    df = tlca.dynamic_inventory_df

    ##### important to convert flow and act as str to excel 
    df["flow"] = df["flow"].astype(str)
    df["activity"] = df["activity"].astype(str)

    ### export df to dp-LCI_output folder, using the act name as the excel name 
    out_dir = "dp-LCI_output/hydro_dpLCI_reservoir"
    os.makedirs(out_dir, exist_ok=True)    
    raw_name = act["name"]
    safe_name = re.sub(r"[^A-Za-z0-9_\-()]+", "_", raw_name)   # replace spaces/special chars
    
    excel_path = os.path.join(out_dir, f"{safe_name}.xlsx")
    
    df.to_excel(excel_path, index=False)    
    print(f"✔ Exported dynamic inventory DF for '{raw_name}' → {excel_path}")


import pickle

out_dir = "dp-LCI_output/hydro_dpLCI_reservoir"
os.makedirs(out_dir, exist_ok=True)

pickle_path = os.path.join(out_dir, "hydro_reservoir_dp_results_MY2030_2050.pkl")

with open(pickle_path, "wb") as f:
    pickle.dump(dp_results, f)

print(f"✔ Saved dp_results dictionary → {pickle_path}")

2026-03-30 10:21:07.316 | INFO     | bw_timex.timex_lca:__init__:114 - Initializing TimexLCA object...
2026-03-30 10:21:07.318 | INFO     | bw_timex.timex_lca:__init__:136 - Collecting node infos...


'market for electricity, hydro, high voltage, RoW, SSP1-VLLO, 2030' (kWh, RoW, None)
TD applied to 'market for electricity, hydro, high voltage, RoW, SSP1-VLLO, 2030' → Exchange: 1 kilowatt hour 'electricity production, hydro, reservoir, non-alpine region' (kilowatt hour, RoW, None) to 'market for electricity, hydro, high voltage, RoW, SSP1-VLLO, 2030' (kWh, RoW, None)>
for the activity 'market for electricity, hydro, high voltage, RoW, SSP1-VLLO, 2030' (kWh, RoW, None),it's under SSP-SSP1-VLLO, year-2030 
 we'll use LCIA ('Climate Change prospective GWP100', 'SSP119', 'MY2030', 'pGWP100 - fixed-AGWPCO2')  with background database.datetime = {'ei_cutoff_3.11_image_SSP1-VLLO_2030 2025-11-24': datetime.datetime(2030, 1, 1, 0, 0), 'ei_cutoff_3.11_image_SSP1-VLLO_2040 2025-11-22': datetime.datetime(2040, 1, 1, 0, 0), 'ei_cutoff_3.11_image_SSP1-VLLO_2050 2025-11-22': datetime.datetime(2050, 1, 1, 0, 0), 'ei_cutoff_3.11_image_SSP2-M_2030 2025-11-22': datetime.datetime(2030, 1, 1, 0, 0), 'ei_

2026-03-30 10:23:47.204 | INFO     | bw_timex.timex_lca:build_timeline:216 - No edge filter function provided. Skipping all edges in background databases.
2026-03-30 10:24:46.900 | INFO     | bw_timex.timex_lca:build_timeline:232 - Calculating base LCA...
2026-03-30 10:24:57.287 | INFO     | bw_timex.timex_lca:build_timeline:242 - Creating activity time mapping...
2026-03-30 10:24:59.818 | INFO     | bw_timex.timeline_builder:__init__:99 - Traversing supply chain graph...


Starting graph traversal


2026-03-30 10:25:01.630 | INFO     | bw_timex.timeline_builder:build_timeline:142 - Building timeline...
2026-03-30 10:25:01.767 | INFO     | bw_timex.timeline_builder:get_weights_for_interpolation_between_nearest_years:522 - Reference date 2026-01-01 00:00:00 is lower than all provided dates. Data will be taken from the closest higher year.
2026-03-30 10:25:01.768 | INFO     | bw_timex.timeline_builder:get_weights_for_interpolation_between_nearest_years:522 - Reference date 2026-01-01 00:00:00 is lower than all provided dates. Data will be taken from the closest higher year.
2026-03-30 10:25:01.769 | INFO     | bw_timex.timeline_builder:get_weights_for_interpolation_between_nearest_years:522 - Reference date 2027-01-01 00:00:00 is lower than all provided dates. Data will be taken from the closest higher year.
2026-03-30 10:25:01.775 | INFO     | bw_timex.timeline_builder:get_weights_for_interpolation_between_nearest_years:522 - Reference date 2028-01-01 00:00:00 is lower than all prov

...

Starting graph traversal


2026-03-30 12:16:54.697 | INFO     | bw_timex.timeline_builder:build_timeline:142 - Building timeline...


Calculation count: 1


2026-03-30 12:16:54.863 | INFO     | bw_timex.timeline_builder:get_weights_for_interpolation_between_nearest_years:522 - Reference date 2026-01-01 00:00:00 is lower than all provided dates. Data will be taken from the closest higher year.
2026-03-30 12:16:54.865 | INFO     | bw_timex.timeline_builder:get_weights_for_interpolation_between_nearest_years:522 - Reference date 2026-01-01 00:00:00 is lower than all provided dates. Data will be taken from the closest higher year.
2026-03-30 12:16:54.866 | INFO     | bw_timex.timeline_builder:get_weights_for_interpolation_between_nearest_years:522 - Reference date 2027-01-01 00:00:00 is lower than all provided dates. Data will be taken from the closest higher year.
2026-03-30 12:16:54.868 | INFO     | bw_timex.timeline_builder:get_weights_for_interpolation_between_nearest_years:522 - Reference date 2028-01-01 00:00:00 is lower than all provided dates. Data will be taken from the closest higher year.
2026-03-30 12:16:54.869 | INFO     | bw_time

0.06826572518730606
0.0682226996752618
✔ Exported dynamic inventory DF for 'market for electricity, hydro, high voltage, RoW, SSP5-H, 2030' → dp-LCI_output/hydro_dpLCI_v2_reservoir/market_for_electricity_hydro_high_voltage_RoW_SSP5-H_2030.xlsx
✔ Saved dp_results dictionary → dp-LCI_output/hydro_dpLCI_reservoir/hydro_reservoir_dp_results_MY2030_2050.pkl
